In [5]:
# %% [code]
!pip install -q datasets indic-nlp-library gensim scikit-learn pandas numpy tqdm

In [6]:
# %% [code]
from datasets import load_dataset
from indicnlp.tokenize import sentence_tokenize
from huggingface_hub import login

from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument

from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

from tqdm import tqdm

import pandas as pd
import numpy as np
import random

In [7]:
# %% [code]
login()

In [8]:
# %% [code]
dataset_dict = load_dataset(
    "ai4bharat/IndicCorpV2",
    streaming=True
)

print("Available Languages:")
print(list(dataset_dict.keys()))

README.md:   0%|          | 0.00/4.01k [00:00<?, ?B/s]

Available Languages:
['asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab', 'khasi', 'santhali']


In [9]:
# %% [code]
LANGUAGE = "hin_Deva"

dataset = dataset_dict[LANGUAGE]

print("Selected Language:", LANGUAGE)

Selected Language: hin_Deva


In [10]:
# %% [code]
TARGET_SENTENCES = 1_200_000

all_sentences = []

for row in tqdm(dataset, desc="Reading Corpus"):

    try:

        sentences = sentence_tokenize.sentence_split(
            row["text"],
            lang="hi"
        )

        for s in sentences:

            s = s.strip()

            if len(s.split()) >= 3:
                all_sentences.append(s)

        if len(all_sentences) >= TARGET_SENTENCES:
            break

    except:
        continue

print("Collected:", len(all_sentences))

Reading Corpus: 763192it [00:29, 25782.46it/s]

Collected: 1200003


In [11]:
# %% [code]
all_sentences = list(dict.fromkeys(all_sentences))

print("Unique Sentences:", len(all_sentences))

Unique Sentences: 1193558


In [12]:
# %% [code]
def stratified_sample(sentences, n, seed=42):

    random.seed(seed)

    lengths = np.array([
        len(s.split()) for s in sentences
    ])

    bins = [0,5,10,15,20,30,40,60,100,np.inf]

    groups = pd.cut(lengths, bins=bins)

    df = pd.DataFrame({
        "sentence": sentences,
        "group": groups
    })

    sampled = []

    proportions = df["group"].value_counts(
        normalize=True,
        sort=False
    )

    for g, p in proportions.items():

        subset = df[df["group"] == g]

        k = min(
            len(subset),
            max(1, round(p*n))
        )

        sampled.append(
            subset.sample(k, random_state=seed)
        )

    result = pd.concat(sampled)

    if len(result) > n:
        result = result.sample(n=n, random_state=seed)

    return result["sentence"].tolist()

In [13]:
# %% [code]
DEV_SIZE = 1000

dev_sentences = stratified_sample(
    all_sentences,
    DEV_SIZE,
    seed=42
)

print("Development:", len(dev_sentences))

Development: 1000


In [14]:
# %% [code]
remaining = [
    s for s in all_sentences
    if s not in set(dev_sentences)
]

test_sentences = stratified_sample(
    remaining,
    1000,
    seed=123
)

print("Test:", len(test_sentences))

Test: 1000


In [15]:
# %% [code]
excluded = set(dev_sentences) | set(test_sentences)

training_pool = [
    s for s in all_sentences
    if s not in excluded
]

random.seed(42)
random.shuffle(training_pool)

train_sentences = training_pool[:1_000_000]

print("Training:", len(train_sentences))
print("Development:", len(dev_sentences))
print("Test:", len(test_sentences))

Training: 1000000
Development: 1000
Test: 1000


In [16]:
# %% [code]
train_tokens = [
    s.split()
    for s in train_sentences
]

dev_tokens = [
    s.split()
    for s in dev_sentences
]

test_tokens = [
    s.split()
    for s in test_sentences
]

print(train_tokens[0][:10])

['अब', 'जगह-जगह', 'से', 'जर्जर', 'हो', 'चुकी', 'है।']


In [ ]:
# %% [code]
from multiprocessing import cpu_count
from tqdm import tqdm
from gensim.models import Word2Vec

word2vec_model = Word2Vec(
    vector_size=100,
    window=5,
    min_count=2,
    workers=cpu_count(),   # Use all available CPU cores
    sg=1
)

# Build vocabulary
print("Building vocabulary...")
word2vec_model.build_vocab(train_tokens)

print("Training Word2Vec...")
epochs = 10

progress = tqdm(range(epochs), desc="Epoch Progress")

for _ in progress:
    word2vec_model.train(
        train_tokens,
        total_examples=word2vec_model.corpus_count,
        epochs=1
    )

print("\nWord2Vec training completed!")
print("Vocabulary Size:", len(word2vec_model.wv))

Building vocabulary...
Training Word2Vec...


Epoch Progress: 100%|██████████| 10/10 [17:01<00:00, 102.16s/it]


Word2Vec training completed!
Vocabulary Size: 177803


In [ ]:
# %% [code]
word = "भारत"

if word in word2vec_model.wv:

    print(f"Most similar words to '{word}':\n")

    for w, score in word2vec_model.wv.most_similar(word, topn=10):
        print(f"{w:15} {score:.4f}")

else:
    print("Word not found.")

Most similar words to 'भारत':

देश             0.7983
चीन             0.7936
पाकिस्तान       0.7910
श्रीलंका        0.7785
अमेरिका         0.7605
बांग्लादेश      0.7572
बांग्‍लादेश     0.7479
जापान           0.7398
रुस             0.7237
(China)         0.7173


In [ ]:
# %% [code]
word2vec_model.save("word2vec.model")

print("Word2Vec model saved successfully.")

Word2Vec model saved successfully.


In [ ]:
import os
import shutil

# Folder in Google Drive
SAVE_DIR = "/content/drive/MyDrive/ILLM_Lab_Assignment5"

os.makedirs(SAVE_DIR, exist_ok=True)

files = [
    "/content/word2vec.model",
    "/content/word2vec.model.wv.vectors.npy",
    "/content/word2vec.model.syn1neg.npy"
]

for file in files:
    shutil.copy(file, SAVE_DIR)

print("Files saved to:", SAVE_DIR)

Files saved to: /content/drive/MyDrive/ILLM_Lab_Assignment5


In [ ]:
# %% [code]
from multiprocessing import cpu_count
from gensim.models import FastText
from tqdm.auto import tqdm

fasttext_model = FastText(
    vector_size=100,
    window=5,
    min_count=2,
    workers=cpu_count(),
    sg=1
)

print("Building FastText vocabulary...")
fasttext_model.build_vocab(corpus_iterable=train_tokens)

epochs = 10

print("Training FastText...")

pbar = tqdm(total=epochs, desc="FastText Epochs")

for epoch in range(epochs):

    fasttext_model.train(
        corpus_iterable=train_tokens,
        total_examples=fasttext_model.corpus_count,
        epochs=1
    )

    pbar.update(1)

pbar.close()

print("\nFastText training completed!")
print("Vocabulary Size:", len(fasttext_model.wv))

Building FastText vocabulary...
Training FastText...


FastText Epochs:   0%|          | 0/10 [00:00<?, ?it/s]


FastText training completed!
Vocabulary Size: 177803


In [ ]:
# %% [code]
word = "भारत"

print(f"Most similar words to '{word}':\n")

for w, score in fasttext_model.wv.most_similar(word, topn=10):
    print(f"{w:15} {score:.4f}")

Most similar words to 'भारत':

’भारत           0.8315
’’भारत          0.8152
भारत!           0.8075
भारत"           0.8066
भारत?           0.7982
चीन-अफगानिस्तान 0.7928
देश             0.7923
पाकिस्तान-भारत  0.7904
पाकिस्तान-अफगानिस्तान 0.7901
अगानिस्तान      0.7901


In [ ]:
# %% [code]
fasttext_model.save("fasttext.model")

print("FastText model saved successfully.")

FastText model saved successfully.


In [ ]:
# %% [code]
import shutil

SAVE_DIR = "/content/drive/MyDrive/ILLM_Lab_Assignment5"

files = [
    "/content/fasttext.model",
    "/content/fasttext.model.wv.vectors_ngrams.npy",
    "/content/fasttext.model.syn1neg.npy",
    "/content/fasttext.model.wv.vectors_vocab.npy"
]

for file in files:
    shutil.copy(file, SAVE_DIR)

print("FastText files saved to Drive.")

FastText files saved to Drive.


In [17]:
# %% [code]
tagged_train = [
    TaggedDocument(words=words, tags=[i])
    for i, words in enumerate(train_tokens)
]

print("Tagged Documents:", len(tagged_train))
print(tagged_train[0])

Tagged Documents: 1000000
TaggedDocument<['अब', 'जगह-जगह', 'से', 'जर्जर', 'हो', 'चुकी', 'है।'], [0]>


In [18]:
# %% [code]
from multiprocessing import cpu_count
from gensim.models import Doc2Vec
from tqdm.auto import tqdm

doc2vec_model = Doc2Vec(
    vector_size=100,
    window=5,
    min_count=2,
    workers=cpu_count(),
    epochs=10,
    dm=1
)

print("Building Doc2Vec vocabulary...")
doc2vec_model.build_vocab(tagged_train)

print("Training Doc2Vec...")

epochs = 10
pbar = tqdm(total=epochs, desc="Doc2Vec Epochs")

for epoch in range(epochs):

    random.shuffle(tagged_train)

    doc2vec_model.train(
        tagged_train,
        total_examples=doc2vec_model.corpus_count,
        epochs=1
    )

    pbar.update(1)

pbar.close()

print("\nDoc2Vec training completed!")

Building Doc2Vec vocabulary...
Training Doc2Vec...


Doc2Vec Epochs:   0%|          | 0/10 [00:00<?, ?it/s]


Doc2Vec training completed!


In [19]:
# %% [code]
test_id = 0

vector = doc2vec_model.infer_vector(train_tokens[test_id])

similar_docs = doc2vec_model.dv.most_similar(
    [vector],
    topn=5
)

print("Original Sentence:\n")
print(train_sentences[test_id])

print("\nMost Similar Sentences:\n")

for doc_id, score in similar_docs:
    print(f"Similarity: {score:.4f}")
    print(train_sentences[doc_id])
    print("-" * 60)

Original Sentence:

अब जगह-जगह से जर्जर हो चुकी है।

Most Similar Sentences:

Similarity: 0.7462
इस बार खेली जाएगी फूलों की होली
------------------------------------------------------------
Similarity: 0.7414
इस बार मां अश्व पर आ रहीं हैं।
------------------------------------------------------------
Similarity: 0.7374
इस बात का पूरा ध्यान रखें कि नहाते वक्त आंखें बंद रखें।
------------------------------------------------------------
Similarity: 0.7352
बाढ़ प्रभावित क्षेत्रों में राहत सामग्री बांटी जा रही हैं।
------------------------------------------------------------
Similarity: 0.7346
अब राहत मिल गई है।
------------------------------------------------------------


In [20]:
# ==========================================
# SAVE COMPLETE DOC2VEC MODEL TO DRIVE
# ==========================================

import os
import shutil

SAVE_DIR = "/content/drive/MyDrive/ILLM_Lab_Assignment5"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save model (this creates both files in /content)
doc2vec_model.save("/content/doc2vec.model")

# Copy both files to Drive
files = [
    "/content/doc2vec.model",
    "/content/doc2vec.model.dv.vectors.npy"
]

for file in files:
    shutil.copy(file, SAVE_DIR)

print("Doc2Vec saved successfully!")

Doc2Vec saved successfully!


In [21]:
import os

MODEL_DIR = "/content/drive/MyDrive/ILLM_Lab_Assignment5"

for f in sorted(os.listdir(MODEL_DIR)):
    if f.startswith("doc2vec"):
        print(f)

doc2vec.model
doc2vec.model.dv.vectors.npy


In [22]:
# %% [code]
import shutil

files = [
    "/content/doc2vec.model"
]

for file in files:
    shutil.copy(file, SAVE_DIR)

print("Doc2Vec saved to Google Drive.")

Doc2Vec saved to Google Drive.


In [23]:
# ==========================================
# CELL 2 : K-MEANS + 20 WORDS PER CLUSTER
# ==========================================

from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

NUM_CLUSTERS = 20

words = word2vec_model.wv.index_to_key
vectors = word2vec_model.wv.vectors

print("Running K-Means...")

kmeans = KMeans(
    n_clusters=NUM_CLUSTERS,
    random_state=42,
    n_init=10
)

labels = kmeans.fit_predict(vectors)
centroids = kmeans.cluster_centers_

print("K-Means completed!\n")

cluster_table = []

for cluster_id in range(NUM_CLUSTERS):

    idx = np.where(labels == cluster_id)[0]

    cluster_vectors = vectors[idx]
    cluster_words = [words[i] for i in idx]

    sims = cosine_similarity(
        [centroids[cluster_id]],
        cluster_vectors
    )[0]

    top = np.argsort(sims)[::-1][:20]

    top_words = [cluster_words[i] for i in top]

    cluster_table.append({
        "Cluster": cluster_id,
        "Top 20 Words": ", ".join(top_words)
    })

cluster_df = pd.DataFrame(cluster_table)

pd.set_option("display.max_colwidth", None)

cluster_df

Running K-Means...
K-Means completed!



,Cluster,Top 20 Words
0,0,"जीतोगे, कडाही, देखने-भालने, ‘राष्ट्रवाद’, छुड़ाते, जनोन्मुखी, सिकायत, पीसेस, रोमेंटिक, डरों, (टीम, Suggestion, low, राजनीति‍, मॉस्चराइज़, Fruit), विलक्षणताओं, फैलाएँ,, आरहा, अदृश्‍य"
1,1,"सिक्सर्स, कॉन्वे, निदहास, सनराइज़र्स, डकेट, Bumrah), रंगना, डे-नाईट, इंडिया-ए, साउथैम्पटन, लेनिंग, जीता|, लॉयन्स, ओवर:, स्‍टोक्‍स, स्टायरिस, मेगान, एसीईसी, मैक्लम, ट्वंटी-20"
2,2,"सकलदीप, बुधलाकोटी,, विलास,, कक्कू, संचारकर्मी, गहरवार,, जाचक,, नगौला, प्रियदर्शिनी,, ,महेश, पैंकरा,, पैगंबरपुर, जगतनारायण, शौरी,, साईलो, बाबेल,, धारासिंह, कालीचरण,, कासलीवाल,, स्वागताध्यक्ष"
3,3,"ओसियन, शेपशफ्टिंग, स्पाइडर-मैन:, वर्ल्ड्स, 1962-63, पेजिंग, Road-, वांग,, आइबीएम,, मोक्सी, एऑन, मैडी, अंतिम-, Nintendo, एनएक्सटी, प्रोफिशिएंसी, विकल्प;, sheet,, यैलो, दिग्गज।"
4,4,"धरदबोचा।, छेडख़ानी, धारधार, छेड़छाड़, क्राइमब्रांच, इनकाउंटर, शोहदे, हत्‍या,, पीटा., गोतस्करों, छुड़वा, नशेबाज, नामजदों, नलिंद, छुड़ाया।, सुखीराम, ईट-पत्थर, दविश, मैदानगढ़ी, पुछताछ"
5,5,"स्वीकृतियों, एजैंडे, हेतु), पारदर्शिता,, एग्जिस्टिंग, कल्याणकारी​, क्रियांवयन, सेवाओ, एकक, पावधान, सावर्जनिक, सहाकरी, षहरों, जनप्रतनिधियों, गोपन, कार्यकरण, गतिवधियों, वित्तपोषण,, प्राथमिक-स्कूल, प्रविधियों"
6,6,"बेपरदा, मांडया, विकासकर्ताओं, आजू-बाजू, गैर-आवासीय, स्‍थानिक, थियानमेन, पुनर्सीमांकन, खटकी, पाम्पलेट, बलारपुर, योगशाला, जलयान, गई;, छेनी, क़ब्रों, भारूवाला, गुशैणी, मंत्रलाय, EHCP"
7,7,"धनवंतरि, पगे, श्रृंगारित, मातृ-पितृ, आदिशंकराचार्य, मत्सय, रूकमणि, सभामंडप, दिवस।, त्रिदेवों, केदारखंड, महानिशा, जगन्नााथ, अर्घ्यदान, चंद्रमौलेश्वर, अनुष्‍ठान, शिलाद, गुरुपूर्णिमा, परब्रह्म, पाश्र्वनाथ"
8,8,"पाडेय,, शाहदेव,, धन्नू,, कहार,, पुनेठा,, प्रेमचन्द्र, चंदोला,, मदनलाल,, कठैत,, चक्रधारी,, मालू,, टोनी,, डॉ.मोहन, सविदर, अंशु,, छिमवाल,, खगेश, चौरे,, ,धर्मेंद्र, कुंभकार,"
9,9,"कल-कारखानों, खाबड़, ट्यूबवैल, उड़कर, टुटू-हीरानगर, हनौता, ल्हासे, एनएच-नौ, लामबगड़, उखड़ी, गिट्टियों, धसकने, पार्र्किंग, बोल्डरों, चढ़ान, गडढे, दूरे, उफना, उफनते, वाहनचालकों"


In [24]:
# ====================================================
# CELL 3 : CLOSEST SENTENCE USING WORD2VEC EMBEDDINGS
# ====================================================

from sklearn.metrics.pairwise import cosine_similarity

def sentence_vector(sentence):

    words = sentence.split()

    vecs = [
        word2vec_model.wv[w]
        for w in words
        if w in word2vec_model.wv
    ]

    if len(vecs) == 0:
        return np.zeros(word2vec_model.vector_size)

    return np.mean(vecs, axis=0)

print("Creating training sentence vectors...")

train_vectors = np.array([
    sentence_vector(s)
    for s in tqdm(train_sentences)
])

def find_closest(query_sentence):

    q = sentence_vector(query_sentence).reshape(1,-1)

    sim = cosine_similarity(q, train_vectors)[0]

    idx = np.argmax(sim)

    return train_sentences[idx], sim[idx]

# -------- Development --------

dev_results = []

for s in tqdm(dev_sentences, desc="Development"):

    nearest, score = find_closest(s)

    dev_results.append([
        s,
        nearest,
        round(float(score),4)
    ])

dev_df = pd.DataFrame(
    dev_results,
    columns=[
        "Validation Sentence",
        "Closest Training Sentence",
        "Similarity"
    ]
)

print("\nValidation Results")
display(dev_df.head(10))

# -------- Test --------

test_results = []

for s in tqdm(test_sentences, desc="Test"):

    nearest, score = find_closest(s)

    test_results.append([
        s,
        nearest,
        round(float(score),4)
    ])

test_df = pd.DataFrame(
    test_results,
    columns=[
        "Test Sentence",
        "Closest Training Sentence",
        "Similarity"
    ]
)

print("\nTest Results")
display(test_df.head(10))

Creating training sentence vectors...


  0%|          | 0/1000000 [00:00<?, ?it/s]

Development:   0%|          | 0/1000 [00:00<?, ?it/s]


Validation Results


,Validation Sentence,Closest Training Sentence,Similarity
0,🔊 पूरी खबर सुने गुरुग्राम।,🔊 पूरी खबर सुने गॉल।,1.0000
1,"मैंने ख़ुद से पूछा, क्या?","मैंने पूछा, कि ना तूने उससे कहा, ना उसने तुझसे कहा, फिर पता कैसे चला, कि प्यार हो गया है?",0.9254
2,पेट पर वसा,"शरीर में आवश्यक पोषक तत्वों की कम मात्रा से पेट से सम्बन्धित गड़बड़ी जैसे दस्त, कब्ज, पाचन शक्ति कमजोर होना, पेट दर्द, या पेट में इन्फेक्शन आदि भी होता हैं।",0.8609
3,घुर्मित भूतल परेउ तुरंता॥,भूतल पर अब केवल शिक्षार्थी लाइसेंस बनेंगे।,0.7186
4,तस्वीरों में देखिये कॉप23.,तस्वीरों में देखिए योगाभ्यास की तस्वीरें-,0.8808
5,रत्न और आभूषण महंगा,"हीरा, रत्न और आभूषण प्रमाणन में हम बाजार में अग्रणी हैं।",0.8160
6,जूते से मारा,जूते से जमकर पीटा,0.8474
7,दुनिया से छुपाने की कोशिश,रंगदारी को पूरी तरह से मिटाने की कोशिश होगी।,0.8916
8,लेकिन बसपा कार्यकर्ताओं को ?,और बीजेपी जो सांगठनिक पार्टी है ।,0.9047
9,. . . ?,. . . !,0.9847


Test:   0%|          | 0/1000 [00:00<?, ?it/s]


Test Results


,Test Sentence,Closest Training Sentence,Similarity
0,"पर, इससे लोड बढ़ेगा।",अब इस फीस पर GST का भार बढने से निश्चित तौर पर एग्जिट लोड बढ़ गया है|,0.8526
1,सफलता के सितारे है ही।,"भाग्य रेखा ही बताती है कि कोई आदमी अपने जीवन में कितनी सफलता हासिल करेगा, कैसा व्यापार या नौकरी करेगा और उसे कब-कब सफलता और असफलता मिलेगी।",0.8891
2,प्लास्टिक की देवी,लोगों ने पूजा के लिए लक्ष्मी गणेश की मूर्तियों के अलावा कुबेर की भी मूर्ति खरीदी।,0.8339
3,आर्थिक मामले अटक सकते हैं।,कानूनी मामलों में फंस सकते हैं।,0.9191
4,यही राफेल की सच्चाई है.,उन्होंने राफेल मामले की सच्चाई उजागर करने की है’।,0.9149
5,मुकाबला काफी मुश्किल था.,ये काफी चैलेंजिंग था.,0.8894
6,ब्रिटेन की कारस्तानी?,ब्रिटेन के अखबार...,0.9057
7,गोशाला में हैं ४५० गाय,पशुपालकों ने डेयरी में गाय और भैंसें रखी हैं।,0.8787
8,बकलोह छावनी में चली गोलियां,नजफगढ़ में गैंगवॉर में रिटायर्ड फौजी की गोलियां बरसाकर जान ली,0.8816
9,पिचिंग की मरम्मत होनी चाहिए।,इनकी मरम्मत होनी है।,0.9222


In [25]:
# ==========================================
# DOC2VEC : CLOSEST SENTENCE (DEV & TEST)
# ==========================================

from tqdm import tqdm
import pandas as pd

def find_doc2vec_closest(sentence):
    vector = doc2vec_model.infer_vector(sentence.split())
    doc_id, score = doc2vec_model.dv.most_similar([vector], topn=1)[0]
    return train_sentences[doc_id], score

# -------- Validation --------
dev_results = []

for s in tqdm(dev_sentences, desc="Validation"):
    nearest, score = find_doc2vec_closest(s)
    dev_results.append([s, nearest, round(float(score), 4)])

dev_doc_df = pd.DataFrame(
    dev_results,
    columns=["Validation Sentence", "Closest Training Sentence", "Similarity"]
)

print("Validation Results (Doc2Vec)")
display(dev_doc_df.head(10))

# -------- Test --------
test_results = []

for s in tqdm(test_sentences, desc="Test"):
    nearest, score = find_doc2vec_closest(s)
    test_results.append([s, nearest, round(float(score), 4)])

test_doc_df = pd.DataFrame(
    test_results,
    columns=["Test Sentence", "Closest Training Sentence", "Similarity"]
)

print("Test Results (Doc2Vec)")
display(test_doc_df.head(10))

Validation: 100%|██████████| 1000/1000 [01:05<00:00, 15.36it/s]

Validation Results (Doc2Vec)


,Validation Sentence,Closest Training Sentence,Similarity
0,🔊 पूरी खबर सुने गुरुग्राम।,प्लेयर ऑफ द मैच घोषित किए गए कप्तान विराट कोहली ने शानदार 85 रनों की पारी खेली।,0.8782
1,"मैंने ख़ुद से पूछा, क्या?",सोशल मीडिया पर उनके लुक की खूब तारीफ भी हुई.,0.8921
2,पेट पर वसा,जहां पर इलाज के दौरान उसकी मौत हो गई।,0.8431
3,घुर्मित भूतल परेउ तुरंता॥,प्लेयर ऑफ द मैच घोषित किए गए कप्तान विराट कोहली ने शानदार 85 रनों की पारी खेली।,0.8695
4,तस्वीरों में देखिये कॉप23.,इस बार मां अश्व पर आ रहीं हैं।,0.8138
5,रत्न और आभूषण महंगा,उन्होंने बताया कि इससे पहले नई दिल्ली का किराया महंगा था।,0.6192
6,जूते से मारा,पुलिस सीसीटीवी की फुटेज से आरोपी की तलाश में लगी है।,0.8377
7,दुनिया से छुपाने की कोशिश,इससे भारत आर्थिक वृद्धि के मोर्चे पर चीन से पिछड़ गया.,0.8710
8,लेकिन बसपा कार्यकर्ताओं को ?,लेकिन समाजवादी कामों को भाजपाई कहां स्वीकार करते हैं।,0.6346
9,. . . ?,अतिक्रमण से मुक्त कराई गई भूमि की अनुमानित कीमत 75 करोड़ रूपए आंकी गई है।,0.9047


Test: 100%|██████████| 1000/1000 [00:57<00:00, 17.43it/s]

Test Results (Doc2Vec)


,Test Sentence,Closest Training Sentence,Similarity
0,"पर, इससे लोड बढ़ेगा।",इससे मुफस्सिल स्थित बस स्टैंड को आने वाले लोगों को दुश्वारियों का सामना करना पड़ा।,0.8267
1,सफलता के सितारे है ही।,"फिल्म में अनुष्का शर्मा, शाहरुख की हीरोइन हैं।",0.7517
2,प्लास्टिक की देवी,हालांकि इससे पहले जैकब की मां ने प्रदर्शनकारियों से शांति बनाए रखने की अपील की थी।,0.8095
3,आर्थिक मामले अटक सकते हैं।,वहीं शाहरुख खान 1992 में फिल्म च्दीवानाज् से डेब्यू किया था।,0.7916
4,यही राफेल की सच्चाई है.,सुप्रीम कोर्ट ने 14 दिसंबर को राफेल से संबंधित याचिकाओं को खारिज किया था।,0.5523
5,मुकाबला काफी मुश्किल था.,23 अगस्त को मतगणना और परिणामों की घोषणा की जाएगी।,0.9024
6,ब्रिटेन की कारस्तानी?,गंभीर मामलों की संख्या घटी,0.7619
7,गोशाला में हैं ४५० गाय,"’’ सिन्हा ने कहा, ‘‘हमारे लिए लक्ष्य यह होना चाहिए कि हम लोगों को अवगत कराएं.",0.8280
8,बकलोह छावनी में चली गोलियां,बेतिया: बिहार के पश्चिमी चंपारण जिले के बेतिया में शुक्रवार को जमकर बवाल हुआ.,0.8013
9,पिचिंग की मरम्मत होनी चाहिए।,इसके अलावा जेसीबी से नालों से भी कचरा निकलवाया गया।,0.8245
